In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 1. Find the repo root (walk upwards until a folder containing src appears)
here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "src").exists()), None)
sys.path.append(str(root / "src"))
import load_system_data as s

# 2. Unload every System Data lorry into one table
df = s.load_all_sd()

# 3. The first and last quarter-hour (UTC), and how many rows we hold
first, last, rows = df["utc"].min(), df["utc"].max(), len(df)

# 4. How many quarter-hours SHOULD exist between first and last
expected = int((last - first) / pd.Timedelta(minutes=15)) + 1

# 5. How many barcodes (utc) appear more than once?
duplicates = int(df["utc"].duplicated().sum())

# 6. Print the verdict
print("first:", first, "| last:", last)
print("rows:", rows, "| expected:", expected, "| gap:", expected - rows)
print("duplicate utc barcodes:", duplicates)

In [ ]:

# 1. Find the repo root and import our loader and config
here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "src").exists()), None)
sys.path.append(str(root / "src"))
import load_system_data as s
import config

# 2. Unload every System Data lorry into one table
df = s.load_all_sd()

# 3. The number columns: every tidy column except the date
num_cols = [c for c in config.SD_RENAME.values() if c != "local_time"]

# 4. The smallest and largest value in each column
print(df[num_cols].agg(["min", "max"]).T)

# 5. How many negative values in each column?
print("\nnegatives per column:")
print((df[num_cols] < 0).sum())

# 6. How many rows have SNSP outside 0 to 1?
print("\nSNSP outside 0-1:", int(((df["snsp"] < 0) | (df["snsp"] > 1)).sum()))

# 7. How many rows have wind generation ABOVE what we would expect?
#    (0.1 MWh allowance for rounding, as in the dispatch-down totals check)
for j in ["ie", "ni"]:
    wind = df[f"{j}_wind_generation"]
    print(f"\n{j.upper()} wind generation above wind availability:",
          int((wind > df[f"{j}_wind_availability"] + 0.1).sum()))
    print(f"{j.upper()} wind generation above total generation:",
          int((wind > df[f"{j}_generation"] + 0.1).sum()))

In [ ]:

# 1. Find the repo root and import our loader
here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "src").exists()), None)
sys.path.append(str(root / "src"))
import load_system_data as s

# 2. Unload every System Data lorry and add a year column to group by
df = s.load_all_sd()
df["year"] = df["utc"].dt.year

# 3. For each jurisdiction, build a small table per year
for j in ["ie", "ni"]:
    gen = df[f"{j}_wind_generation"]
    avail = df[f"{j}_wind_availability"]

    summary = pd.DataFrame({
        "rows": df.groupby("year").size(),
        # rows where generation is above availability (0.1 allowance for rounding)
        "gen_above_avail": (gen > avail + 0.1).groupby(df["year"]).sum(),
        # rows where availability is exactly zero
        "avail_is_zero": (avail == 0).groupby(df["year"]).sum(),
        # the biggest amount by which generation beat availability that year
        "largest_excess": (gen - avail).groupby(df["year"]).max().round(1),
    })
    print(j.upper())
    print(summary)
    print()